# 🍽️ Food Calorie AI
موديل بيشوف صورة طبق أكل ويقدّر نوعه وكميته وسعراته.

- **الجزء الأول:** جرّب الموديل على طول (أقل من دقيقة، مش محتاج GPU).
- **الجزء التاني (اختياري):** تدريب الموديل من الأول (محتاج T4 GPU، حوالي ساعة ونص).

# الجزء الأول: جرّب على طول 🚀

In [ ]:
# 1) نجيب المشروع
import os
REPO_URL = 'https://github.com/Abdallaallam5/food-calorie-ai.git'
if not os.path.exists('/content/food-calorie-ai'):
    !git clone -q {REPO_URL} /content/food-calorie-ai
else:
    !cd /content/food-calorie-ai && git pull -q
%cd /content/food-calorie-ai
!pip install -q timm

### 2) ارفع صورة أكل وشوف النتيجة
- صوّر الطبق **من فوق** وهو باين كله، والأفضل طبق سادة.
- الموديل بيتعرف على 101 صنف عالمي (بيتزا، برجر، سوشي، ستيك، سلطات، حلويات...).
- `SIZE = None` معناها إن الكمية بتتقدّر من الصورة. أو اختار `'small'` / `'medium'` / `'large'`.

In [ ]:
SIZE = None  # None = تقدير من الصورة | 'small' / 'medium' / 'large'

import matplotlib.pyplot as plt
from google.colab import files
from PIL import Image
from src.predict import CalorieEstimator, format_result
from src.portion import visualize

est = CalorieEstimator()
for name in files.upload():
    img = Image.open(name)
    result, portion = est.estimate(img, size=SIZE, return_portion=True)

    n = 2 if portion else 1
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    axes = list(axes) if n == 2 else [axes]
    axes[0].imshow(img); axes[0].set_title('Input'); axes[0].axis('off')
    if portion:
        axes[1].imshow(visualize(portion)); axes[1].axis('off')
        axes[1].set_title(f'Plate (green) / Food (orange): {portion.coverage:.0%}')
    plt.show()
    print(format_result(result))

---
# الجزء التاني (اختياري): تدريب الموديل من الأول
قبل ما تبدأ: **Runtime → Change runtime type → T4 GPU**، وشغّل خلية الجزء الأول (1) الأول.

In [ ]:
# نحفظ على Drive عشان لو الجلسة فصلت ما نخسرش التدريب
from google.colab import drive
drive.mount('/content/drive')
os.environ['SAVE_DIR'] = '/content/drive/MyDrive/food_calorie_model'
os.environ['DATA_DIR'] = '/content/data'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'مفيش GPU! غيّر الـ runtime')

In [ ]:
# نستكشف الداتا (أول مرة بتحمّل ~5GB)
import random
from src.data import get_datasets

train_raw, test_raw = get_datasets(with_transforms=False)
print(f'Train: {len(train_raw)} | Test: {len(test_raw)} | Classes: {len(train_raw.classes)}')

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax in axes.flat:
    img, label = train_raw[random.randint(0, len(train_raw) - 1)]
    ax.imshow(img); ax.set_title(train_raw.classes[label]); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# التدريب (حوالي ساعة لساعة ونص). لو الجلسة فصلت هيكمل من آخر epoch.
!python -m src.train